# Chapter 3 — Clear Notebooks, Visible Data, and No Magic

Chapter 2 built a complete miniature language-model pipeline.

This chapter focuses on how to make that kind of work inspectable and trustworthy.

By the end of the chapter, you will be able to:

- keep each code cell focused on one purpose;
- reveal important text and data without flooding the notebook;
- use descriptive names and readable training examples;
- distinguish assertions from input validation;
- control randomness with local seeded generators;
- make debug output optional; and
- recognize and prevent hidden state.

The central standard is simple:

> A reader should be able to see what the data is, how it changes, and which assumptions must remain true.

## Why Inspectability Matters

Language-model bugs often look plausible.

A notebook may run while using a shifted target, a mismatched vocabulary, an unexpected newline, or a variable left behind by an earlier execution.

Clear notebooks reduce those risks by making key facts visible:

- raw text and hidden characters;
- tokens and token IDs;
- input-target alignment;
- important invariants; and
- the source of randomness.

The solution is not more abstraction.

The solution is to expose the right information at the right time.

## Terms and the Running Fixture

A **fixture** is a small, fixed piece of local data used for teaching or testing.

An **assertion** stops execution when an internal assumption is false.

**Deterministic** code produces the same result when its inputs and environment are unchanged.

**Hidden state** is information that affects a notebook but is not established by reading and running its cells from top to bottom.

We will reuse the Chapter 2 fixture so the data is familiar:

```text
the dog ran
the cat sat
```

## Inspect Raw Text Before Transforming It

A normal print shows the text as a reader sees it.

`repr` reveals invisible characters such as the leading, internal, and trailing newlines.

In [1]:
raw_text = """
the dog ran
the cat sat
"""

print("Raw text:")
print(raw_text)
print("Raw text with hidden characters visible:")
print(repr(raw_text))

Raw text:

the dog ran
the cat sat

Raw text with hidden characters visible:
'\nthe dog ran\nthe cat sat\n'


## Use Small Preview Helpers

Large values can overwhelm a notebook when printed in full.

A useful preview helper shows the object's name, size, and a small representative sample.

The helper should remove output repetition without hiding the transformation being taught.

In [2]:
from collections.abc import Sequence


def print_text_preview(name: str, text: str, max_characters: int = 80) -> None:
    if max_characters < 1:
        raise ValueError("max_characters must be at least 1")
    preview = text[:max_characters]
    print(f"{name}: {len(text)} characters")
    print("  visible preview:", preview)
    print("  repr preview:   ", repr(preview))


def print_sequence_preview(
    name: str,
    values: Sequence[object],
    max_items: int = 8,
) -> None:
    if max_items < 1:
        raise ValueError("max_items must be at least 1")
    preview = list(values[:max_items])
    print(f"{name}: {len(values)} items")
    print(f"  first {len(preview)}:", preview)


print_text_preview("raw_text", raw_text)

raw_text: 25 characters
  visible preview: 
the dog ran
the cat sat

  repr preview:    '\nthe dog ran\nthe cat sat\n'


## Keep Transformations Focused

The next cells repeat only the minimum pipeline needed for this chapter's inspection examples.

Each cell performs one conceptual transformation and displays its result before moving on.

In [3]:
cleaned_text = raw_text.strip()

print_text_preview("before cleaning", raw_text)
print_text_preview("after cleaning", cleaned_text)

before cleaning: 25 characters
  visible preview: 
the dog ran
the cat sat

  repr preview:    '\nthe dog ran\nthe cat sat\n'
after cleaning: 23 characters
  visible preview: the dog ran
the cat sat
  repr preview:    'the dog ran\nthe cat sat'


## Use Descriptive Names

Names such as `tokens`, `token_to_id`, and `target_token_id` expose the role of each value.

Names such as `x`, `t2i`, and `ys` force a beginner to memorize private shorthand.

Short mathematical names may become useful later when their meaning is conventional and local.

The data pipeline benefits from explicit names.

In [4]:
def tokenize_words(text: str) -> list[str]:
    return text.split()


tokens = tokenize_words(cleaned_text)
vocabulary = sorted(set(tokens))
token_to_id = {token: token_id for token_id, token in enumerate(vocabulary)}
id_to_token = {token_id: token for token, token_id in token_to_id.items()}
print_sequence_preview("tokens", tokens)
print_sequence_preview("vocabulary", vocabulary)
print("token_to_id:", token_to_id)
print("id_to_token:", id_to_token)

tokens: 6 items
  first 6: ['the', 'dog', 'ran', 'the', 'cat', 'sat']
vocabulary: 5 items
  first 5: ['cat', 'dog', 'ran', 'sat', 'the']
token_to_id: {'cat': 0, 'dog': 1, 'ran': 2, 'sat': 3, 'the': 4}
id_to_token: {0: 'cat', 1: 'dog', 2: 'ran', 3: 'sat', 4: 'the'}


## Assert Important Invariants

An **invariant** is a condition that must remain true for later results to be trustworthy.

Encoding and decoding should preserve the token sequence exactly.

Assertions are appropriate for internal notebook invariants.

They should not replace explicit validation for function arguments because Python can disable assertions in optimized mode.

In [5]:
def encode_tokens(tokens: list[str], token_to_id: dict[str, int]) -> list[int]:
    return [token_to_id[token] for token in tokens]


def decode_token_ids(
    token_ids: list[int],
    id_to_token: dict[int, str],
) -> list[str]:
    return [id_to_token[token_id] for token_id in token_ids]


token_ids = encode_tokens(tokens, token_to_id)
decoded_tokens = decode_token_ids(token_ids, id_to_token)
assert decoded_tokens == tokens
print_sequence_preview("token_ids", token_ids)
print_sequence_preview("decoded_tokens", decoded_tokens)
print("Token round trip succeeds:", decoded_tokens == tokens)

token_ids: 6 items
  first 6: [4, 1, 2, 4, 0, 3]
decoded_tokens: 6 items
  first 6: ['the', 'dog', 'ran', 'the', 'cat', 'sat']
Token round trip succeeds: True


## Print Training Examples Readably

Chapter 2 represented each example as one pair containing context IDs and a target ID.

Keeping the values paired prevents separate input and target lists from drifting out of alignment.

The builder validates arguments with `ValueError` and the notebook asserts the expected example count afterward.

In [6]:
def build_next_token_examples(
    token_ids: list[int],
    context_length: int,
) -> list[tuple[list[int], int]]:
    if context_length < 1:
        raise ValueError("context_length must be at least 1")
    if context_length >= len(token_ids):
        raise ValueError("context_length must be smaller than the token sequence")

    examples: list[tuple[list[int], int]] = []
    for start_position in range(len(token_ids) - context_length):
        target_position = start_position + context_length
        input_token_ids = token_ids[start_position:target_position]
        examples.append((input_token_ids, token_ids[target_position]))
    return examples


context_length = 2
training_examples = build_next_token_examples(token_ids, context_length)
expected_example_count = len(token_ids) - context_length
assert len(training_examples) == expected_example_count
print("Expected examples:", expected_example_count)
print("Actual examples:", len(training_examples))

Expected examples: 4
Actual examples: 4


In [7]:
def print_training_examples(
    examples: list[tuple[list[int], int]],
    id_to_token: dict[int, str],
) -> None:
    for example_number, (input_ids, target_id) in enumerate(examples, start=1):
        input_tokens = decode_token_ids(input_ids, id_to_token)
        target_token = id_to_token[target_id]
        print(
            f"Example {example_number}: input={input_tokens}, target={target_token!r}"
        )


print_training_examples(training_examples, id_to_token)

Example 1: input=['the', 'dog'], target='ran'
Example 2: input=['dog', 'ran'], target='the'
Example 3: input=['ran', 'the'], target='cat'
Example 4: input=['the', 'cat'], target='sat'


## Control Randomness Locally

Stored teaching examples should not change silently between runs.

A local `random.Random` instance isolates the example from unrelated global randomness.

A fixed seed makes the pseudorandom sequence reproducible in the same supported environment.

It does not make the underlying choice non-random as a concept.

In [8]:
import random


def sample_tokens(
    possible_tokens: list[str],
    probabilities: list[float],
    sample_count: int,
    random_seed: int,
) -> list[str]:
    random_generator = random.Random(random_seed)
    return random_generator.choices(
        possible_tokens, weights=probabilities, k=sample_count
    )


possible_tokens = ["cat", "dog", "ran"]
probabilities = [0.2, 0.3, 0.5]
first_samples = sample_tokens(possible_tokens, probabilities, 10, random_seed=7)
second_samples = sample_tokens(possible_tokens, probabilities, 10, random_seed=7)
assert first_samples == second_samples
print("First run:", first_samples)
print("Second run:", second_samples)
print("Runs match:", first_samples == second_samples)

First run: ['dog', 'cat', 'ran', 'cat', 'ran', 'dog', 'cat', 'ran', 'cat', 'dog']
Second run: ['dog', 'cat', 'ran', 'cat', 'ran', 'dog', 'cat', 'ran', 'cat', 'dog']
Runs match: True


## Make Debug Output Optional

Detailed output is useful while inspecting a tiny example.

The same output becomes noise when a training loop repeats thousands of times.

A debug flag should control presentation without changing the returned data.

In [9]:
def inspect_examples(
    examples: list[tuple[list[int], int]],
    id_to_token: dict[int, str],
    *,
    debug: bool,
) -> None:
    if not debug:
        return
    print("Debug example details:")
    print_training_examples(examples, id_to_token)


inspect_examples(training_examples, id_to_token, debug=True)
print("Calling with debug=False produces no details.")
inspect_examples(training_examples, id_to_token, debug=False)
print("Quiet call completed.")

Debug example details:
Example 1: input=['the', 'dog'], target='ran'
Example 2: input=['dog', 'ran'], target='the'
Example 3: input=['ran', 'the'], target='cat'
Example 4: input=['the', 'cat'], target='sat'
Calling with debug=False produces no details.
Quiet call completed.


## Describe Data Structure Honestly

Later chapters will inspect array and tensor shapes frequently.

Python lists do not guarantee a rectangular shape because rows may have different lengths.

Separate vector and matrix helpers make that distinction explicit and report ragged rows instead of inventing a shape.

In [10]:
def describe_vector(name: str, values: list[int]) -> None:
    print(f"{name}: length {len(values)}")


def describe_matrix(name: str, rows: list[list[int]]) -> None:
    if not rows:
        print(f"{name}: empty matrix")
        return
    row_lengths = [len(row) for row in rows]
    if len(set(row_lengths)) == 1:
        print(f"{name}: {len(rows)} rows × {row_lengths[0]} columns")
    else:
        print(f"{name}: ragged rows with lengths {row_lengths}")


input_rows = [input_ids for input_ids, _ in training_examples]
describe_vector("token_ids", token_ids)
describe_matrix("input_rows", input_rows)
describe_matrix("ragged_example", [[1, 2], [3]])

token_ids: length 6
input_rows: 4 rows × 2 columns
ragged_example: ragged rows with lengths [2, 1]


## Handle Expected Errors Explicitly

An instructional notebook should not contain unexplained error outputs.

When an error is the lesson, catch it and label why it was expected.

The next cell demonstrates both invalid context-length boundaries without stopping the notebook.

In [11]:
invalid_context_lengths = [0, len(token_ids)]
for invalid_length in invalid_context_lengths:
    try:
        build_next_token_examples(token_ids, invalid_length)
    except ValueError as error:
        print(f"context_length={invalid_length}: {error}")

context_length=0: context_length must be at least 1
context_length=6: context_length must be smaller than the token sequence


## Avoid Hidden State

A notebook has hidden state when its visible order does not establish everything needed for execution.

A common symptom is a notebook that works only after cells are run in a special order.

Use these safeguards:

- place each import in the first cell that needs it;
- define fixtures and functions before use;
- avoid redefining the same name with a different meaning;
- use local files and declared dependencies; and
- execute from a fresh kernel before considering the notebook complete.

## Know When Not to Plot

A plot belongs in a notebook only when it makes an important relationship easier to understand.

Chapter 2 already plotted the relationship between context length and example count.

Repeating that plot here would add decoration rather than information.

This chapter intentionally has no plot.

Choosing not to visualize is part of good notebook design.

## Final Notebook Check

A final cell can collect the small number of invariants that protect the chapter's conclusions.

It should confirm important state without rerunning or hiding the pipeline.

In [12]:
assert cleaned_text == "the dog ran\nthe cat sat"
assert decoded_tokens == tokens
assert len(training_examples) == len(token_ids) - context_length
assert first_samples == second_samples
print("Final checks passed.")
print("Tokens:", len(tokens))
print("Vocabulary entries:", len(vocabulary))
print("Training examples:", len(training_examples))

Final checks passed.
Tokens: 6
Vocabulary entries: 5
Training examples: 4


## Key Takeaways

- Show data at the point where it changes.
- Use concise previews instead of either hiding data or printing everything.
- Prefer descriptive names while concepts are still new.
- Use assertions for internal invariants and exceptions for invalid inputs.
- Keep stored randomness deterministic with local seeded generators.
- Make verbose debug output optional.
- Report ragged lists honestly instead of pretending they have a rectangular shape.
- Run every notebook from a fresh kernel to expose hidden state.
- Omit plots that do not teach a new relationship.

## Next Chapter

The next chapter returns to text itself.

It will examine characters, spaces, newlines, punctuation, and finite sequences more carefully because every later model depends on those details being represented correctly.